In [ ]:
!pip install xlsxwriter
import pandas as pd
import numpy as np
from datetime import datetime
from google.colab import files
from google.colab import drive; drive.mount('/content/drive')

# ---------------------------------------------------------
# CONFIG archivo Excel
# ---------------------------------------------------------

archivo = r"/content/drive/MyDrive/Sales BILANNATURALS 2023-Mar2026.xlsm.xlsx"
hoja = "Reporte Ventas"

# ---------------------------------------------------------
# 1. Cargar datos
# ---------------------------------------------------------
df = pd.read_excel(archivo, sheet_name=hoja)
df.columns = df.columns.str.lower().str.strip()

df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors="coerce")

# Convert 'total' column to numeric, coercing errors to NaN
df["total"] = pd.to_numeric(df["total"], errors="coerce")
df = df.dropna(subset=["fecha", "cliente", "total"])

# ---------------------------------------------------------
# 2. CONSOLIDAR COMPRAS REALES (cliente + fecha = una compra)
# ---------------------------------------------------------
df_compra = (
    df.groupby(["cliente", "fecha"])
    .agg(
        total_compra=("total", "sum"),
        familias=("familia", "nunique"),
        items=("cantidad", "sum")).reset_index())

df_compra = df_compra.sort_values(["cliente", "fecha"])

# ---------------------------------------------------------
# 3. Calcular FRECUENCIA REAL DE COMPRA
# ---------------------------------------------------------
df_compra["delta_dias"] = (
    df_compra.groupby("cliente")["fecha"]
    .diff().dt.days)

frecuencia = (
    df_compra.groupby("cliente")["delta_dias"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={
        "mean": "frecuencia_promedio",
        "std": "frecuencia_std"}))

frecuencia = frecuencia.dropna()

# Redondear promedio
frecuencia["frecuencia_promedio"] = (
    frecuencia["frecuencia_promedio"]
    .round().astype(int))

# Rellenar std NaN (clientes muy constantes)
frecuencia["frecuencia_std"] = (
    frecuencia["frecuencia_std"].fillna(0))

# ---------------------------------------------------------
# 4. CLASIFICAR FRECUENCIA POR RANGOS
# ---------------------------------------------------------
bins = [0, 30, 60, 90, 120, 150, 10000]
labels = [
    "1-30 dias",
    "31-60 dias",
    "61-90 dias",
    "91-120 dias",
    "121-150 dias",
    "150+ dias"]

frecuencia["rango_frecuencia"] = pd.cut(
    frecuencia["frecuencia_promedio"],
    bins=bins,
    labels=labels,
    right=True)

# ---------------------------------------------------------
# 5. OUTPUT FINAL POR CLIENTE
# ---------------------------------------------------------
ultima_compra = (
    df_compra.groupby("cliente")["fecha"]
    .max()
    .reset_index()
    .rename(columns={"fecha": "ultima_fecha_compra"}))

resultado_final = ultima_compra.merge(
    frecuencia,on="cliente",how="inner")

today = datetime.today()

# Días desde última compra
resultado_final["dias_desde_ultima"] = (
    today - resultado_final["ultima_fecha_compra"]).dt.days

# Próxima compra estimada
resultado_final["prox_compra_calculada"] = (
    resultado_final["ultima_fecha_compra"]
    - pd.Timedelta(days=3)
    + pd.to_timedelta(
        resultado_final["frecuencia_promedio"],unit="D"))

# SCORE DINÁMICO DE RECOMPRA
resultado_final["score_recompra"] = (
    resultado_final["dias_desde_ultima"] /
    resultado_final["frecuencia_promedio"])

# VENTANA PROBABILISTICA STD
resultado_final["limite_inferior"] = (
    resultado_final["frecuencia_promedio"]
    - resultado_final["frecuencia_std"])

resultado_final["limite_superior"] = (
    resultado_final["frecuencia_promedio"]
    + resultado_final["frecuencia_std"])

# ---------------------------------------------------------
# 5.1 FEATURES DE VALOR POR CLIENTE
# ---------------------------------------------------------

df_valor = (
    df_compra.groupby("cliente")
    .agg(
        total_promedio=("total_compra", "mean"),
        total_std=("total_compra", "std"),
        cantidad_promedio=("items", "mean"),
        num_familias=("familias", "nunique")).reset_index())

# limpiar std
df_valor["total_std"] = df_valor["total_std"].fillna(0)

# merge con resultado_final
resultado_final = resultado_final.merge(df_valor,on="cliente",how="left")

# ---------------------------------------------------------
# 5.2 FAMILIA MÁS COMPRADA
# ---------------------------------------------------------

familia_top = (df.groupby(["cliente", "familia"]).size()
    .reset_index(name="conteo"))

familia_top = familia_top.sort_values(
    ["cliente", "conteo"],ascending=False).drop_duplicates("cliente")

resultado_final = resultado_final.merge(
    familia_top[["cliente", "familia"]],on="cliente",how="left")

# ---------------------------------------------------------
# 5.3 VARIACIÓN DE GASTO
# ---------------------------------------------------------

resultado_final["variacion_gasto"] = (
    resultado_final["total_std"] /
    resultado_final["total_promedio"])

resultado_final["variacion_gasto"] = (
    resultado_final["variacion_gasto"]
    .replace([np.inf, -np.inf], 0).fillna(0))

# ---------------------------------------------------------
# 5.4 PROBABILIDAD DE RECOMPRA
# ---------------------------------------------------------

# score
resultado_final["prob_recompra"] = resultado_final["score_recompra"]

# ajustar con variabilidad (clientes inestables ↓ probabilidad)
resultado_final["prob_recompra"] = (
    resultado_final["prob_recompra"] *(1 - resultado_final["variacion_gasto"]))

# normalizar entre 0 y 1
resultado_final["prob_recompra"] = (
    resultado_final["prob_recompra"].clip(0, 1))


# ---------------------------------------------------------
# 5.5 VALOR ESPERADO DE RECOMPRA
# ---------------------------------------------------------

resultado_final["valor_estimado"] = (
    resultado_final["total_promedio"] *
     (1 + resultado_final["variacion_gasto"]))

# ---------------------------------------------------------
# 5.6 SCORE FINAL
# ---------------------------------------------------------

resultado_final["score_final"] = (
    resultado_final["prob_recompra"] *
    resultado_final["valor_estimado"])

# ---------------------------------------------------------
# 6. FILTRAR CLIENTES POR MES Y AÑO OBJETIVO
# ---------------------------------------------------------

"""today = datetime.today()

# Días desde última compra
resultado_final["dias_desde_ultima"] = (
    today - resultado_final["ultima_fecha_compra"]).dt.days

# SCORE DINÁMICO
resultado_final["score_recompra"] = (
    resultado_final["dias_desde_ultima"] /
    resultado_final["frecuencia_promedio"])

# FILTRO POR ZONA DE RECOMPRA
resultado_filtrado = resultado_final[
    (resultado_final["score_recompra"] >= 0.40) &
    (resultado_final["score_recompra"] <= 1.50)].copy()

# Ordenar por urgencia
resultado_filtrado = resultado_filtrado.sort_values(
    "score_recompra",ascending=False)

# Validación
print("Clientes filtrados:", resultado_filtrado.shape[0])
print("Filtro basado en score dinámico (0.40 - 1.50)")"""

# ---------------------------------------------------------
# 6. FILTRO FINAL INTELIGENTE
# ---------------------------------------------------------

resultado_filtrado = resultado_final[
    (resultado_final["prob_recompra"] >= 0.35) &
    (resultado_final["score_final"] > resultado_final["score_final"].quantile(0.50))].copy()

#ordenar por impacto económico
resultado_filtrado = resultado_filtrado.sort_values(
    "score_final",ascending=False)

# validación
print("Clientes filtrados:", resultado_filtrado.shape[0])
print("Filtro basado en probabilidad + valor esperado")

# ---------------------------------------------------------
# 7. Exportar CSV FINAL
# ---------------------------------------------------------

fecha_hoy = datetime.now().strftime("%d-%m-%Y")
nombre_excel = f"CLIENTES PROX A COMPRAR {fecha_hoy}.xlsx"

resultado_filtrado = resultado_filtrado.sort_values("prox_compra_calculada")

resultado_filtrado = resultado_filtrado.sort_values("score_final",ascending=False)

with pd.ExcelWriter(nombre_excel) as writer:
    resultado_filtrado.to_excel(writer, index=False)

print("Archivo generado correctamente:", nombre_excel)
print("Total clientes exportados:", resultado_filtrado.shape[0])
files.download(nombre_excel)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clientes filtrados: 664
Filtro basado en probabilidad + valor esperado
Archivo generado correctamente: CLIENTES PROX A COMPRAR 07-04-2026.xlsx
Total clientes exportados: 664


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ahora, vamos a revisar cada una de las condiciones de filtrado para ver cuántos clientes cumplen cada una: